<a href="https://colab.research.google.com/github/leejaewon23/AI_dev/blob/main/%EC%9E%90%EC%97%B0_%EC%9E%A5%EB%A9%B4_%EB%B6%84%EB%A5%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

데이터셋 다운로드

In [ ]:
# Kaggle API 설치 및 데이터 다운로드
!pip install kaggle -q

# kaggle.json 업로드 (Kaggle 사이트 → 계정 → API → Create New Token)
from google.colab import files
files.upload()  # kaggle.json 파일 선택

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 데이터셋 다운로드
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip -q intel-image-classification.zip -d ./data

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
License(s): copyright-authors
100% 346M/346M [00:09<00:00, 39.2MB/s]



라이브러리 및 장치 설정

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

사용 장치: cuda


데이터 전처리

In [ ]:
# 학습 데이터 증강
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),          # Intel 이미지는 150x150 → 64x64로 통일
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(64, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],   # ImageNet 기준값 사용
                         [0.229, 0.224, 0.225])
])

# 테스트 데이터 (증강 없이 정규화만)
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# 데이터셋 로드 (폴더 구조 자동 인식)
train_dataset = datasets.ImageFolder('./data/seg_train/seg_train', transform=train_transform)
test_dataset  = datasets.ImageFolder('./data/seg_test/seg_test',   transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

classes = train_dataset.classes
print(f"클래스 목록: {classes}")       # ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
print(f"학습 데이터: {len(train_dataset)}장")
print(f"테스트 데이터: {len(test_dataset)}장")

클래스 목록: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
학습 데이터: 14034장
테스트 데이터: 3000장


CNN 모델 설계

In [ ]:
class Intel_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)        # 64x64 → 32x32
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)        # 32x32 → 16x16
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)        # 16x16 → 8x8
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # 8x8 → 1x1
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 6)         # 6개 클래스 출력
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

model = Intel_CNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {total_params:,}")

파라미터 수: 1,214,534


학습 설정 및 실행

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += outputs.max(1)[1].eq(labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset) * 100

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            correct += outputs.max(1)[1].eq(labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset) * 100


EPOCHS = 50
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9,
                      weight_decay=5e-4, nesterov=True)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-4)

best_acc = 0
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc   = evaluate(model, test_loader, criterion, device)
    scheduler.step()

    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}% | "
          f"Test Loss: {test_loss:.4f} Acc: {test_acc:.1f}%")

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'best_intel_model.pth')

print(f"\n최고 테스트 정확도: {best_acc:.2f}%")

Epoch  1/50 | Train Loss: 1.2361 Acc: 57.1% | Test Loss: 1.0370 Acc: 68.9%
Epoch  2/50 | Train Loss: 0.9953 Acc: 73.7% | Test Loss: 1.0419 Acc: 71.8%
Epoch  3/50 | Train Loss: 0.9059 Acc: 78.9% | Test Loss: 0.9083 Acc: 75.7%
Epoch  4/50 | Train Loss: 0.8527 Acc: 81.1% | Test Loss: 0.8481 Acc: 80.5%
Epoch  5/50 | Train Loss: 0.8160 Acc: 83.1% | Test Loss: 0.7995 Acc: 83.4%
Epoch  6/50 | Train Loss: 0.7851 Acc: 84.4% | Test Loss: 0.7607 Acc: 85.4%
Epoch  7/50 | Train Loss: 0.7643 Acc: 85.8% | Test Loss: 0.7394 Acc: 85.9%
Epoch  8/50 | Train Loss: 0.7450 Acc: 86.4% | Test Loss: 0.7470 Acc: 84.8%
Epoch  9/50 | Train Loss: 0.7348 Acc: 87.0% | Test Loss: 0.7434 Acc: 85.2%
Epoch 10/50 | Train Loss: 0.7249 Acc: 87.4% | Test Loss: 0.7411 Acc: 85.1%
Epoch 11/50 | Train Loss: 0.7113 Acc: 88.1% | Test Loss: 0.8316 Acc: 80.6%
Epoch 12/50 | Train Loss: 0.7056 Acc: 88.1% | Test Loss: 0.8289 Acc: 81.6%
Epoch 13/50 | Train Loss: 0.6981 Acc: 88.7% | Test Loss: 0.7320 Acc: 85.5%
Epoch 14/50 | Train Loss: